In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import pandas as pd
import sys

# Get the absolute path to the target file's directory
scraper_dir = os.path.abspath('/Users/tylermartins/Documents/PitchSights/ps-betting/strategies/anytime_goalscorer/models')
sys.path.append(scraper_dir)

# Now import from the file (without the .py extension)
import ags_model as ags


In [2]:
# Paths to reloaded files
processed_data_path = '/Users/tylermartins/Documents/PitchSights/ps-betting/strategies/anytime_goalscorer/data/processed'

# Leagues:
# Premier-League - 9
# La-Liga - 12
# Serie-A - 11
# Ligue-1
# Bundesliga - 20

season = '2024-2025'
league = 'Premier-League'

df_cleaned = pd.read_csv(f"{processed_data_path}/features_{league}_{season}.csv")


# Step 3: Convert match_date to datetime and sort
df_cleaned['match_date'] = pd.to_datetime(df_cleaned['match_date'])
df_cleaned = df_cleaned.sort_values(by=['match_date', 'gameweek_number'])

# Step 4: Drop non-feature columns
columns_to_drop = [
    'match_date', 'player', 'bookmaker',
       'team', 'position', 'age_decimal', 'minutes', 'goals', 'shots',
       'shots_on_target', 'xg', 'npxg', 'xg_assist', 'sca', 'gca',
       'miscontrols', 'dispossessed', 'passes_received',
       'progressive_passes_received', 'take_ons', 'take_ons_won',
       'take_ons_tackled', 'touches_att_3rd', 'touches_att_pen_area',
       'team_shots', 'team_shots_on_target', 'team_xG', 'home_team',
       'away_team', 'opp', 'opp_xG', 'team_score', 'opp_score',
       'team_possession', 'opp_possession', 'team_passing_accuracy',
       'opp_passing_accuracy', 'opp_shots_on_target', 'opp_shots',
       'team_saves', 'opp_saves', 'team_saves_faced', 'opp_saves_faced',
       'gameweek_number', 'shots_on_target_ma', 'goals_ma', 'shots_ma',
       'xg_ma', 'npxg_ma', 'xg_assist_ma', 'sca_ma', 'gca_ma',
       'touches_att_3rd_ma', 'touches_att_pen_area_ma', 'miscontrols_ma',
       'dispossessed_ma', 'passes_received_ma',
       'progressive_passes_received_ma', 'take_ons_won_ma',
       'take_ons_tackled_ma','model_confidence',
    'target_hit_line'  # this is the label, will be used in the model script
]
feature_cols = [col for col in df_cleaned.columns if col not in columns_to_drop]


In [3]:
feature_cols

['ags_odds',
 'pens_att',
 'minutes_ma',
 'goals_per90_ma',
 'sot_per90_ma',
 'shots_per90_ma',
 'xg_per90_ma',
 'npxg_per90_ma',
 'xa_per90_ma',
 'sca_per90_ma',
 'gca_per90_ma',
 'touches_att_third_per90_ma',
 'touches_att_pen_area_per90_ma',
 'miscontrols_per90_ma',
 'dispossessed_per90_ma',
 'passes_received_per90_ma',
 'progressive_passes_received_per90_ma',
 'take_ons_won_per90_ma',
 'take_ons_tackled_per90_ma',
 'team_shots_ma',
 'team_shots_on_target_ma',
 'team_xG_ma',
 'sot_team_share',
 'shots_team_share',
 'xg_team_share',
 'implied_prob_ags',
 'is_home',
 'opponent_sot_allowed_per90',
 'opponent_xg_allowed_per90',
 'is_penalty_taker']

In [4]:
# Choose model type once
model_type = "xgboost"
gameweeks = sorted(df_cleaned['gameweek_number'].unique())
all_results = []
all_importance = []

for gw in gameweeks[3:]:
    print(f"Training and predicting for Gameweek {gw}")
    results, importance = ags.ags_model(df_cleaned,feature_cols, gw, model_type=model_type)
    all_results.append(results)
    all_importance.append(importance.set_index('feature'))

# Combine prediction results
final_results = pd.concat(all_results, ignore_index=True)

# Compute average feature importance across gameweeks
avg_importance_df = pd.concat(all_importance, axis=1).mean(axis=1).reset_index()
avg_importance_df.columns = ['feature', 'avg_importance']
avg_importance_df = avg_importance_df.sort_values(by='avg_importance', ascending=False)

# Show top 10 features
print("\n🔍 Top 10 Predictive Features (Avg. Importance):")
print(avg_importance_df.head(10))


Training and predicting for Gameweek 7
Training and predicting for Gameweek 8
Training and predicting for Gameweek 9


/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Training and predicting for Gameweek 10
Training and predicting for Gameweek 11
Training and predicting for Gameweek 12


/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Training and predicting for Gameweek 13
Training and predicting for Gameweek 14
Training and predicting for Gameweek 15


...

/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:19] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:19] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:19] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Training and predicting for Gameweek 36
Training and predicting for Gameweek 37


/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:19] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:19] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Training and predicting for Gameweek 38

🔍 Top 10 Predictive Features (Avg. Importance):
                    feature  avg_importance
1                  ags_odds        0.145468
3                  pens_att        0.077225
0          is_penalty_taker        0.066838
28            npxg_per90_ma        0.056250
16              xg_per90_ma        0.045251
2   team_shots_on_target_ma        0.038090
5                team_xG_ma        0.036663
14               minutes_ma        0.035525
8             xg_team_share        0.030800
10         shots_team_share        0.030377


/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [5]:
final_results


,player,match_date,gameweek_number,bookmaker,ags_odds,goals,actual,predicted,probability
0,Bukayo Saka,2024-10-05,7,FanDuel,2.3,1,1,1,0.717213
1,Declan Rice,2024-10-05,7,BetRivers,5.4,0,0,0,0.083326
2,Gabriel Jesus,2024-10-05,7,BetMGM,2.5,0,0,0,0.292005
3,Gabriel Magalhães,2024-10-05,7,FanDuel,6.5,0,0,0,0.112121
4,Gabriel Martinelli,2024-10-05,7,BetMGM,2.6,1,1,1,0.554828
...,...,...,...,...,...,...,...,...,...
7082,Matt Doherty,2025-05-25,38,BetRivers,10.0,0,0,0,0.023485
7083,Nélson Semedo,2025-05-25,38,FanDuel,18.0,0,0,0,0.036416
7084,Rayan Aït-Nouri,2025-05-25,38,FanDuel,8.0,0,0,0,0.061880
7085,Santiago Bueno,2025-05-25,38,Bovada,18.0,0,0,0,0.021110


In [6]:
ev_threshold = 0.08

final_results['implied_prob_over'] = 1 / final_results['ags_odds']

final_results['ev'] = final_results['probability'] - final_results['implied_prob_over']


plus_ev = final_results[(final_results['ev'] > 0)&(final_results['predicted'] == 1)]
plus_ev_wins = final_results[(final_results['ev'] > 0)&(final_results['predicted'] == 1)&(final_results['actual'] == 1)]

ev_plus_co = final_results[(final_results['ev'] > ev_threshold)&(final_results['predicted'] == 1)]
ev_plus_co_wins = final_results[(final_results['ev'] > ev_threshold)&(final_results['predicted'] == 1)&(final_results['actual'] == 1)]


print(f'+EV Bets: {len(plus_ev)}')
print(f'+EV Wins: {len(plus_ev_wins)}')
print(f"+EV Win Rate: {len(plus_ev_wins) / len(plus_ev)}")

print(f"+EV Bets Avg. EV: {plus_ev['ev'].mean()}")
print(f"+EV Wins Avg. EV: {plus_ev_wins['ev'].mean()}")

print(f'EV + {ev_threshold*100}% Bets: {len(ev_plus_co)}')
print(f'EV + {ev_threshold*100}% Wins: {len(ev_plus_co_wins)}')
print(f"EV + {ev_threshold*100}% Win Rate: {len(ev_plus_co_wins) / len(ev_plus_co)}")

# print(f"+EV Bets Avg. EV: {plus_ev['ev'].mean()}")
# print(f"+EV Wins Avg. EV: {plus_ev_wins['ev'].mean()}")


+EV Bets: 112
+EV Wins: 70
+EV Win Rate: 0.625
+EV Bets Avg. EV: 0.31712179956314607
+EV Wins Avg. EV: 0.37377719141764154
EV + 8.0% Bets: 103
EV + 8.0% Wins: 67
EV + 8.0% Win Rate: 0.6504854368932039


In [7]:
final_results[final_results['probability']>=0.6]

final_results[(final_results['ev'] > ev_threshold)&(final_results['predicted'] == 1)&(final_results['probability'] >= 0.6)]

,player,match_date,gameweek_number,bookmaker,ags_odds,goals,actual,predicted,probability,implied_prob_over,ev
0,Bukayo Saka,2024-10-05,7,FanDuel,2.30,1,1,1,0.717213,0.434783,0.282430
25,Bryan Mbeumo,2024-10-05,7,FanDuel,2.85,1,1,1,0.782363,0.350877,0.431485
90,Anthony Gordon,2024-10-05,7,BetRivers,3.05,0,0,1,0.796568,0.327869,0.468699
138,Cole Palmer,2024-10-06,7,FanDuel,2.20,0,0,1,0.660959,0.454545,0.206414
216,Bryan Mbeumo,2024-10-19,8,FanDuel,3.70,0,0,1,0.693709,0.270270,0.423439
...,...,...,...,...,...,...,...,...,...,...,...
6402,Eberechi Eze,2025-05-05,35,DraftKings,5.00,1,1,1,0.653872,0.200000,0.453872
6480,Danny Welbeck,2025-05-10,36,DraftKings,3.30,1,1,1,0.826289,0.303030,0.523259
6719,Bryan Mbeumo,2025-05-18,37,FanDuel,2.75,1,1,1,0.875071,0.363636,0.511435
7005,Erling Haaland,2025-05-25,38,FanDuel,1.83,1,1,1,0.917879,0.546448,0.371431


In [10]:
ev_plus_co.to_csv(f'/Users/tylermartins/Documents/PitchSights/ps-betting/strategies/anytime_goalscorer/outputs/ags_model_output_{league}_{season}.csv')

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import pandas as pd
import numpy as np

# Choose model type: "random_forest" or "xgboost"
model_type = "xgboost"  # or "random_forest"

# Prepare features and target
X = df_cleaned[feature_cols]
y = df_cleaned['target_hit_line']
gameweeks = sorted(df_cleaned['gameweek_number'].unique())

all_results = []
importance_cumulative = np.zeros(len(feature_cols))  # 🔍 to track average importance

for i in range(3, len(gameweeks)):
    print("Training Week:", gameweeks[i])
    train_weeks = gameweeks[:i]
    test_week = gameweeks[i]

    train_idx = df_cleaned['gameweek_number'].isin(train_weeks)
    test_idx = df_cleaned['gameweek_number'] == test_week

    X_train, y_train = X[train_idx], y[train_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Select model
    if model_type == "random_forest":
        model = RandomForestClassifier(n_estimators=100, random_state=42)
    elif model_type == "xgboost":
        model = xgb.XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            use_label_encoder=False,
            n_estimators=100,
            max_depth=3,
            learning_rate=0.1,
            random_state=42
        )
    else:
        raise ValueError("Invalid model_type. Choose 'random_forest' or 'xgboost'.")

    # Train and predict
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    # 🔍 Accumulate feature importance
    importance_cumulative += model.feature_importances_

    # Store results
    results_df = df_cleaned.loc[test_idx, [
        'player', 'match_date', 'gameweek_number', 'ags_odds', 'goals'
    ]].copy()
    results_df['actual'] = y_test.values
    results_df['predicted'] = y_pred
    results_df['probability'] = y_proba

    all_results.append(results_df)

# Combine all results
final_results = pd.concat(all_results, ignore_index=True)

# 🔍 Average feature importances
average_importance = importance_cumulative / (len(gameweeks) - 3)
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'avg_importance': average_importance
}).sort_values(by='avg_importance', ascending=False)

# Show top 10 most predictive features
print("\n🎯 Top 10 Most Predictive Features:")
print(importance_df.head(10))


Training Week: 7
Training Week: 8
Training Week: 9


/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Training Week: 10
Training Week: 11
Training Week: 12


/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Training Week: 13
Training Week: 14
Training Week: 15


...

/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Training Week: 34
Training Week: 35
Training Week: 36


/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:22] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:23] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Training Week: 37
Training Week: 38

🎯 Top 10 Most Predictive Features:
                    feature  avg_importance
0                  ags_odds        0.145468
1                  pens_att        0.077225
29         is_penalty_taker        0.066838
7             npxg_per90_ma        0.056250
6               xg_per90_ma        0.045251
20  team_shots_on_target_ma        0.038090
21               team_xG_ma        0.036663
2                minutes_ma        0.035525
24            xg_team_share        0.030800
23         shots_team_share        0.030377


/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:23] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:16:23] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
